# 12 · External validation of the Nudge engine and app

**Why:** Steps 1-10 validated Nudge on the data it was built with. This notebook tests it from the outside: on data where the
true answer is known (Part A), on four public uplift benchmarks against outside methods (Part B), on bad and unusual uploads
(Part C) and for stability (Part D). It tests the **app's own code** (`app/lib/planner.py`, `app/lib/upload.py`,
`app/nudge_engine.py`), so a pass means the deployed app works.

**Rules:** the pass marks were written in `VALIDATION_PLAN.md` before any run and are not changed. Fixed seeds. Every result has a
95% range (bootstrap, 200 repeats). Raw rows stay in `data_benchmarks/` (git-ignored); only aggregates are saved.

**How it runs:** the heavy work lives in small helper files in `validation/` (one per part). Each part below re-runs its helper
only if its cached results (`validation/cache/*.json`) are missing, so this notebook opens in seconds; delete the cache to re-run
from scratch (about 1.5 hours, mostly Part B's causal forest). Environment: `validation/.venv` (kernel "Nudge validation").

In [1]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'validation'))
import pandas as pd
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 30); pd.set_option('display.precision', 3)
import common, scorecard

## Part A · Known truth (synthetic customers)

**What:** 12 made-up campaign tests where we *planted* the true effect for every customer: 30% buy anyway, the message adds
+4 pts on average (SD 4 pts), about 16% are put off by it. 4-arm files have 3 options with different costs (SMS ₹0.10,
WhatsApp ₹1.02, a ₹20 discount paid by buyers), each helping different customers.

**Why:** on real data nobody knows each customer's true response, so we can only score rankings. Here we can check directly:
does the engine rank customers in the right order (A1, Spearman), and how much of the best possible profit does the app's plan
earn (A2, plan ÷ oracle)? A3 feeds it data with **no effect at all**: a good tool must then say "do not personalise".

Pass marks: A1 Spearman ≥ 0.5 on each 50k file; A2 ≥ 70% of oracle on each 50k and 2 lakh file; A3 ≥ 18 of 20.

In [2]:
if not common.load('A3_null'):
    import runpy; runpy.run_path(str(common.ROOT / 'validation' / 'run_part_a.py'), run_name='__main__')
A = scorecard.part_a('A'); A

,rows,options,outcome,A1 Spearman,A1 low95,A1 high95,A2 plan/oracle,A2 low95,A2 high95,plan chosen,simple plan (truth)/oracle,A2 at Rs1.02 (info)
0,5000,1,binary,0.485,0.433,0.532,0.534,0.463,0.603,Nudge plan,0.254,0.774
1,5000,1,revenue,0.532,0.492,0.570,0.318,0.198,0.415,Contact everyone (Message),0.318,0.868
2,5000,3,binary,0.495,0.465,0.521,0.463,0.437,0.485,Contact everyone (SMS reminder),0.530,NaN
3,5000,3,revenue,0.217,0.182,0.253,0.542,0.509,0.569,Contact everyone (WhatsApp reminder),0.542,NaN
4,50000,1,binary,0.633,0.621,0.646,0.322,0.286,0.356,Contact everyone (Message),0.322,0.877
5,50000,1,revenue,0.483,0.468,0.500,0.589,0.570,0.604,Nudge plan,0.335,0.883
6,50000,3,binary,0.474,0.464,0.484,0.657,0.649,0.667,Nudge plan,0.532,NaN
7,50000,3,revenue,0.429,0.418,0.439,0.674,0.665,0.682,Nudge plan,0.526,NaN
8,200000,1,binary,0.792,0.788,0.796,0.847,0.843,0.853,Nudge plan,0.343,0.901
9,200000,1,revenue,0.789,0.784,0.793,0.846,0.840,0.850,Nudge plan,0.335,0.934


In [3]:
a50, a2l = A[A.rows == 50_000], A[A.rows == 200_000]
print('A1 at 50k  :', 'PASS' if (a50['A1 Spearman'] >= 0.5).all() else 'FAIL', f"({(a50['A1 Spearman'] >= 0.5).sum()} of 4 files >= 0.5)")
print('A2 at 50k+ :', 'PASS' if (pd.concat([a50, a2l])['A2 plan/oracle'] >= 0.7).all() else 'FAIL',
      f"(50k: {(a50['A2 plan/oracle'] >= 0.7).sum()} of 4, 2 lakh: {(a2l['A2 plan/oracle'] >= 0.7).sum()} of 4)")
N = scorecard.null('A')
print(f"A3 null    : {'PASS' if N['not_personalised'] >= 18 else 'FAIL'} ({N['not_personalised']} of 20 'do not personalise'; "
      f"95% range of the rate {N['low95_rate']:.2f} to {N['high95_rate']:.2f})")
N['table'].T

A1 at 50k  : FAIL (1 of 4 files >= 0.5)
A2 at 50k+ : FAIL (50k: 0 of 4, 2 lakh: 4 of 4)


A3 null    : FAIL (17 of 20 'do not personalise'; 95% range of the rate 0.62 to 0.97)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
seed,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20
personalise,False,False,True,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True,False
recommended,Contact nobody,Contact everyone (Message),Nudge plan,Contact nobody,Contact nobody,Contact everyone (Message),Nudge plan,Contact everyone (Message),Contact everyone (Message),Contact nobody,Contact everyone (Message),Contact everyone (Message),Contact nobody,Contact nobody,Contact everyone (Message),Contact nobody,Contact nobody,Contact nobody,Nudge plan,Contact nobody


### Why Part A failed at 50k: the diagnosis

For each 50k file: the **true** share of the oracle profit that each plan the planner could choose would earn, and what the
16% validation slice (8,000 customers) said about it. The engine's personalised plans are close to 70% of the oracle, but the
slice is too noisy to pick between them and "contact everyone"; on the 2-arm binary file it picked "everyone" (32%) over a
personalised plan (70%).

In [4]:
import diagnose_a, run_part_a
r = diagnose_a.candidates(50_000, 2, 'binary', run_part_a.seed_of(50_000, 2, 'binary'))
print('chosen by the app:', r.attrs['chosen'], '| learned margin', r.attrs['learned_margin']); r

chosen by the app: Contact everyone (Message) | learned margin 0.25


,truth_ratio,val_value,share
plan,,,
margin x0,0.698,49.361,0.489
margin x0.1,0.698,49.325,0.489
margin x0.25,0.697,49.378,0.488
margin x0.5,0.697,49.331,0.488
margin x1,0.697,49.359,0.487
margin x2,0.695,49.246,0.485
margin xinf,0.000,45.449,0.000
everyone (Message),0.322,50.428,1.000
nobody,0.000,45.449,0.000


### Run 2: a candidate fix (tested here only, the app is NOT changed)

**Candidate:** decide between plans with 5-fold cross-fitting on the whole 80% instead of one 16% slice, and train the final
engine on the 80% (`validation/candidate_fix.py`). Settings were chosen on separate development seeds, then run **once** on the
official seeds. Both runs are kept.

In [5]:
R2 = scorecard.part_a('run2')
if len(R2):
    cmp = A.merge(R2, on=['rows', 'options', 'outcome'], suffixes=(' run1', ' run2'))
    display(cmp[['rows', 'options', 'outcome', 'A1 Spearman run1', 'A1 Spearman run2', 'A2 plan/oracle run1', 'A2 plan/oracle run2',
                 'A2 low95 run2', 'A2 high95 run2', 'plan chosen run1', 'plan chosen run2']])
    N2 = scorecard.null('run2')
    if N2: print(f"A3 null run 2: {N2['not_personalised']} of {N2['runs']} 'do not personalise'")

,rows,options,outcome,A1 Spearman run1,A1 Spearman run2,A2 plan/oracle run1,A2 plan/oracle run2,A2 low95 run2,A2 high95 run2,plan chosen run1,plan chosen run2
0,50000,1,binary,0.633,0.665,0.322,0.728,0.711,0.742,Contact everyone (Message),Nudge plan (margin x0)
1,50000,1,revenue,0.483,0.510,0.589,0.629,0.611,0.646,Nudge plan,Nudge plan (margin x1)
2,50000,3,binary,0.474,0.517,0.657,0.747,0.740,0.755,Nudge plan,Nudge plan (margin x0.5)
3,50000,3,revenue,0.429,0.443,0.674,0.731,0.722,0.738,Nudge plan,Nudge plan (margin x1)
4,200000,1,binary,0.792,0.817,0.847,0.862,0.857,0.866,Nudge plan,Nudge plan (margin x1)
5,200000,1,revenue,0.789,0.821,0.846,0.871,0.866,0.875,Nudge plan,Nudge plan (margin x0.1)
6,200000,3,binary,0.691,0.749,0.862,0.896,0.893,0.897,Nudge plan,Nudge plan (margin x0)
7,200000,3,revenue,0.630,0.673,0.852,0.876,0.873,0.879,Nudge plan,Nudge plan (margin x0)


A3 null run 2: 17 of 20 'do not personalise'


## Part B · Public benchmarks vs outside methods

**What:** Criteo (ads, 10% sample), Lenta (grocery SMS), MegaFon (telecom), Hillstrom (e-mail). Locked 20% test; every method
trains on the same 2 lakh rows of the rest. Methods: random order, likely-buyer model (the usual approach), the Nudge fast engine
(app default), the Nudge ensemble (thorough mode), econml's CausalForestDML and R-learner.

**Why:** a tool that only beats random proves little. Here it competes with well-known outside methods on data it never saw.

Pass marks (judged on Qini only): B1 engine Qini 95% range above 0 on every dataset with a real effect; B2 engine within 10% of
the best method on ≥ 3 of 4 datasets.

In [6]:
B = scorecard.part_b()
marks = scorecard.b_marks(B); marks

,real effect (ATE range > 0),ATE,ATE low95,ATE high95,engine Qini,low95,high95,B1 pass,best method,best Qini,engine / best,B2 within 10%,ensemble / best (info)
dataset,,,,,,,,,,,,,
criteo10,True,0.010,0.009,0.011,0.068,0.047,0.089,True,econml CausalForestDML,0.090,0.759,False,0.954
lenta,True,0.008,0.006,0.009,-0.001,-0.014,0.013,False,Likely-buyer model,0.011,-0.113,False,0.148
megafon,True,0.050,0.047,0.052,0.216,0.208,0.224,True,Nudge fast engine,0.216,1.000,True,0.959
hillstrom,True,0.045,0.039,0.052,0.064,0.026,0.093,True,econml CausalForestDML,0.068,0.942,True,0.955


In [7]:
for d, (r, t) in B.items():
    print(f"\n{d}: {r['description']}\n  {r['rows_total']:,} rows; train {r['rows_train']:,}; locked test {r['rows_test']:,}; "
          f"treated {r['treated_share']:.0%}; outcome rate {r['outcome_rate']:.2%}; app fairness light: {r['fairness']} (largest gap {r['max_gap']:.3f})")
    cols = ['qini', 'qini_low95', 'qini_high95', 'auuc', 'top10_lift', 'top30_lift', 'profit_top30_per_customer', 'engine_minus_this_qini',
            'engine_minus_this_qini_low95', 'engine_minus_this_qini_high95']
    display(t[[c for c in cols if c in t]].drop(index='Treat everyone').astype(float).round(4))


criteo10: Criteo (ads): 10% random sample (seed 2026) of the official 13.98M-row file; treatment = could be shown ads; outcome = site visit
  1,398,073 rows; train 200,000; locked test 279,615; treated 85%; outcome rate 4.69%; app fairness light: green (largest gap 0.044)


,qini,qini_low95,qini_high95,auuc,top10_lift,top30_lift,profit_top30_per_customer,engine_minus_this_qini,engine_minus_this_qini_low95,engine_minus_this_qini_high95
Random order,-4.000e-04,-0.015,0.013,-2.000e-04,0.010,0.010,0.160,0.069,0.045,0.092
Likely-buyer model,8.940e-02,0.071,0.108,3.520e-02,0.058,0.031,1.073,-0.021,-0.044,-0.002
Nudge fast engine,6.830e-02,0.047,0.089,2.690e-02,0.048,0.027,0.893,NaN,NaN,NaN
Nudge ensemble (thorough),8.580e-02,0.067,0.105,3.380e-02,0.055,0.028,0.947,-0.018,-0.032,-0.003
econml CausalForestDML,8.990e-02,0.069,0.109,3.540e-02,0.054,0.030,1.035,-0.022,-0.041,-0.002
econml R-learner (NonParamDML),6.720e-02,0.044,0.090,2.650e-02,0.054,0.028,0.965,0.001,-0.014,0.015



lenta: Lenta (Russian grocery): SMS vs no SMS; outcome = came and bought (response_att)
  687,029 rows; train 200,000; locked test 137,406; treated 75%; outcome rate 10.82%; app fairness light: red (largest gap 0.193)


,qini,qini_low95,qini_high95,auuc,top10_lift,top30_lift,profit_top30_per_customer,engine_minus_this_qini,engine_minus_this_qini_low95,engine_minus_this_qini_high95
Random order,-1.210e-02,-2.200e-02,-1.000e-04,-6.600e-03,-0.002,8.000e-04,-0.271,1.090e-02,-0.007,0.027
Likely-buyer model,1.070e-02,-1.000e-03,2.240e-02,5.800e-03,0.023,1.210e-02,0.236,-1.190e-02,-0.028,0.005
Nudge fast engine,-1.200e-03,-1.390e-02,1.340e-02,-7.000e-04,0.010,8.500e-03,0.074,NaN,NaN,NaN
Nudge ensemble (thorough),1.600e-03,-9.400e-03,1.450e-02,8.000e-04,0.015,1.030e-02,0.158,-2.800e-03,-0.014,0.009
econml CausalForestDML,8.500e-03,-8.000e-04,1.940e-02,4.700e-03,0.016,1.010e-02,0.148,-9.700e-03,-0.027,0.006
econml R-learner (NonParamDML),-7.000e-04,-1.330e-02,1.420e-02,-4.000e-04,0.002,1.010e-02,0.147,-5.000e-04,-0.013,0.013



megafon: MegaFon (telecom, synthetic-like features X_1..X_50): treatment vs control; outcome = conversion
  600,000 rows; train 200,000; locked test 120,000; treated 50%; outcome rate 20.42%; app fairness light: green (largest gap 0.007)


,qini,qini_low95,qini_high95,auuc,top10_lift,top30_lift,profit_top30_per_customer,engine_minus_this_qini,engine_minus_this_qini_low95,engine_minus_this_qini_high95
Random order,0.010,0.003,0.016,0.007,0.057,0.057,2.235,2.067e-01,1.980e-01,0.216
Likely-buyer model,0.147,0.140,0.154,0.101,0.249,0.152,6.527,6.980e-02,6.330e-02,0.076
Nudge fast engine,0.216,0.208,0.224,0.149,0.366,0.200,8.694,NaN,NaN,NaN
Nudge ensemble (thorough),0.207,0.200,0.215,0.143,0.344,0.195,8.465,8.900e-03,7.100e-03,0.011
econml CausalForestDML,0.206,0.200,0.213,0.142,0.360,0.198,8.607,1.020e-02,5.700e-03,0.015
econml R-learner (NonParamDML),0.216,0.208,0.224,0.149,0.371,0.202,8.766,4.000e-04,-9.000e-04,0.002



hillstrom: Hillstrom: women's e-mail vs no e-mail; outcome = site visit in 2 weeks
  42,693 rows; train 34,154; locked test 8,539; treated 50%; outcome rate 12.88%; app fairness light: green (largest gap 0.009)


,qini,qini_low95,qini_high95,auuc,top10_lift,top30_lift,profit_top30_per_customer,engine_minus_this_qini,engine_minus_this_qini_low95,engine_minus_this_qini_high95
Random order,0.015,-0.021,0.054,0.008,0.064,0.060,2.397,4.910e-02,8.000e-04,0.092
Likely-buyer model,0.055,0.016,0.084,0.025,0.088,0.078,3.192,8.600e-03,-2.660e-02,0.037
Nudge fast engine,0.064,0.026,0.093,0.028,0.060,0.074,3.033,NaN,NaN,NaN
Nudge ensemble (thorough),0.065,0.026,0.096,0.028,0.066,0.082,3.373,-9.000e-04,-1.000e-02,0.009
econml CausalForestDML,0.068,0.027,0.098,0.030,0.081,0.073,2.963,-3.900e-03,-2.190e-02,0.015
econml R-learner (NonParamDML),0.059,0.022,0.091,0.026,0.062,0.076,3.131,4.300e-03,-5.900e-03,0.013


In [8]:
# calibration by decile (methods that predict an uplift size): average predicted vs measured uplift; slope 1 = well calibrated
for d, (r, t) in B.items():
    for m in ['Nudge fast engine', 'econml CausalForestDML', 'econml R-learner (NonParamDML)']:
        c = t.loc[m, 'calibration']
        print(f"{d:9s} {m:32s} slope {c['slope_measured_on_predicted']:+.2f}")

criteo10  Nudge fast engine                slope +0.57
criteo10  econml CausalForestDML           slope +0.85
criteo10  econml R-learner (NonParamDML)   slope +nan
lenta     Nudge fast engine                slope -0.04
lenta     econml CausalForestDML           slope +0.51
lenta     econml R-learner (NonParamDML)   slope -0.11
megafon   Nudge fast engine                slope +1.30
megafon   econml CausalForestDML           slope +1.58
megafon   econml R-learner (NonParamDML)   slope +1.32
hillstrom Nudge fast engine                slope +0.50
hillstrom econml CausalForestDML           slope +0.85
hillstrom econml R-learner (NonParamDML)   slope +0.48


## Part C · Bad and unusual uploads

**What:** 12 files from the plan's list plus 4 extras, each pushed through the app's real upload path (read, column guess,
checks, preparation, fairness light, train, plan, Money page curves, the download, thorough mode) and, as a second layer, through
the real pages run headless (Streamlit AppTest through `app.py`).

**Why:** real users upload messy files. The app must never crash, must say clearly what is wrong, and must refuse to pretend a
non-random file is a fair test. Pass marks: C1 0 crashes; C2 clear messages; C3 non-random file red; C4 1 lakh rows planned in < 60 s.
**No app code was changed**: failures are logged with a proposed fix for review.

In [9]:
Cr = common.load('C_robustness')
rows = []
for k, v in Cr.items():
    rows.append({'file': k, 'what': v['description'], 'result': v['stopped_at'], 'fairness': (v.get('fairness') or {}).get('light'),
                 'plan': (v.get('plan') or {}).get('recommended'), 'messages': ' | '.join(f'{l}: {m}' for l, m in v['messages'])[:300],
                 'crash': (v['crash'] or {}).get('error')})
pd.DataFrame(rows).set_index('file')

,what,result,fairness,plan,messages,crash
file,,,,,,
missing_values,"15% of each measure missing, 5% of a text meas...",CRASH at train,green,NaN,warning: 1.9% of rows have no usable outcome a...,ValueError: Input data must be 2 dimensional a...
text_columns,"text measures: city (60 values), gender, tier ...",finished (plan shown),green,Contact everyone (Message),,NaN
extra_wrong_names,"index column, odd names, control called ""Holdo...",finished (plan shown),green,Contact nobody,,NaN
outcome_all_zero,nobody bought anything,finished (plan shown),green,Contact nobody,,NaN
rows_500,500 rows,finished (plan shown),green,Contact everyone (Message),"warning: Only 257 customers got ""No message"". ...",NaN
split_95_5,"95% got the message, 5% control (about 500 peo...",finished (plan shown),green,Contact nobody,"warning: Only 486 customers got ""No message"". ...",NaN
non_random,the message went to the top half by a spend-li...,finished (plan shown),red,Contact everyone (Message),,NaN
duplicate_ids,"2,000 customers appear twice (10%)",finished (plan shown),green,Contact everyone (Message),,NaN
options_4plus,control + 4 options,finished (plan shown),green,Nudge plan,,NaN


In [10]:
T = common.load('C_apptest') or {}
pd.DataFrame([{'file': k, 'stages': ' > '.join(s[0] for s in v.get('stages', [])),
               'page exceptions': sum(len(s[1].get('exceptions', [])) for s in v.get('stages', [])),
               'first exception': next((s[1]['exceptions'][0][:200] for s in v.get('stages', []) if s[1].get('exceptions')), None)}
              for k, v in T.items()]).set_index('file')

,stages,page exceptions,first exception
file,,,
rows_500,loaded > built (after rerun) > views/money.py ...,1,The `value` 0 is less than the `min_value` 100...
missing_values,loaded > built (after rerun) > views/money.py ...,1,Input data must be 2 dimensional and non empty...
text_columns,loaded > built (after rerun) > views/money.py ...,0,NaN
extra_wrong_names,loaded > built (after rerun) > views/money.py ...,0,NaN
extra_wrong_names_userfixed,loaded > built (after rerun) > views/money.py ...,0,NaN
outcome_all_zero,loaded > built (after rerun) > views/money.py ...,0,NaN
split_95_5,loaded > built (after rerun) > views/money.py ...,0,NaN
non_random,loaded > train anyway ticked > built (after re...,0,NaN
duplicate_ids,loaded > built (after rerun) > views/money.py ...,0,NaN


In [11]:
t4 = common.load('C4_timing')
if t4: print(f"C4: 1,00,000 rows, upload to plan {t4['total_s']:.1f} s (read {t4['read_s']:.1f}, prepare {t4['prepare_s']:.1f}, "
             f"train {t4['train_s']:.1f}, plan {t4['plan_s']:.1f}) -> {'PASS' if t4['total_s'] < 60 else 'FAIL'}")

C4: 1,00,000 rows, upload to plan 17.0 s (read 0.2, prepare 0.4, train 13.4, plan 2.9) -> PASS

## Part D · Stability

**What:** the same 50k file trained 5 times, changing only the models' random seed. **Why:** if the top-20% target list changed a
lot between runs, a marketing team could not trust it. Pass mark: average overlap ≥ 70%.

In [12]:
D = common.load('D_stability')
print(f"D1: average top-20% overlap {D['average_overlap']:.3f} (95% range {D['average_low95']:.3f} to {D['average_high95']:.3f}); "
      f"lowest pair {D['lowest_pair']} {D['lowest_overlap']:.3f} -> {'PASS' if D['pass'] else 'FAIL'}")
pd.Series(D['pairs'], name='overlap').to_frame().T

D1: average top-20% overlap 0.819 (95% range 0.809 to 0.827); lowest pair 2-4 0.809 -> PASS


,1-2,1-3,1-4,1-5,2-3,2-4,2-5,3-4,3-5,4-5
overlap,0.816,0.821,0.819,0.823,0.817,0.809,0.823,0.815,0.825,0.824


## Scorecard and what it means
See `VALIDATION_REPORT.md` for the scorecard, the plain summary, the Part C failure list with proposed fixes, and the ranked
list of weaknesses.